# ChE 210 - Excel Clinic 1 demo with LLM and Python


Reads triplicate outlet-concentration measurements from a CSV, averages the
trials at each residence time, plots the means with standard-deviation error
bars for each reactor temperature, fits a straight line to each data set, and
saves a publication-quality JPEG.

----
**AI Citation:**
An LLM was used to generate the following plotting script:

---
*"Using the attached csv file, create a Python script using matplotlib to create a publication-quality plot complete with title ("Reactor concentration"), axes labels (x= "Residence time [min]", y="Outlet Concentration [M]"), and legend for each temperature.* 

*Average over the three trials and include error bars of the std dev at each data point.* 

*Include human readable comments. Use a san serif font, tight layout, and save as a jpeg format with dpi = 300. Perform a linear regression on the provided data and report the coefficients."*

Version 1: Sonnet, ver. 5 Medium; Anthropic: San Francisco, CA, June 30, 2026.
https://claude.ai/share/bbbd0250-a190-4569-b490-8d9153db527e (accessed 2026-08-24).

Version 2: Opus, ver. 5 High; Anthropic: San Francisco, CA, July 24, 2026.
https://claude.ai/share/3bc9edb7-a0a0-43d1-b45a-f11d374a5814 (accessed 2026-08-24).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ----------------------------------------------------------------------
# 1. User settings
# ----------------------------------------------------------------------
CSV_PATH = "ChE_210_Excel_Clinic_1_Template.csv"
OUT_PATH = "reactor_concentration.jpg"

# The raw file has several merged/blank header rows above the numbers, so the
# data block is read directly and the columns are named by hand.
HEADER_ROWS = 5          # number of lines to skip before the first data row
TEMPERATURES = [300, 400]  # K, in the order the trial blocks appear

# ----------------------------------------------------------------------
# 2. Load the data
# ----------------------------------------------------------------------
# Column layout of the data block:
#   0: blank spacer column from Excel
#   1: residence time (min)
#   2-4: T = 300 K, trials 1-3
#   5-7: T = 400 K, trials 1-3
raw = pd.read_csv(CSV_PATH, skiprows=HEADER_ROWS, header=None)

tau = raw.iloc[:, 1].astype(float).to_numpy()          # residence time [min]
trials = {                                             # 3-column block per temperature
    300: raw.iloc[:, 2:5].astype(float).to_numpy(),
    400: raw.iloc[:, 5:8].astype(float).to_numpy(),
}

In [ ]:

# ----------------------------------------------------------------------
# 3. Statistics: mean and sample standard deviation across the three trials
# ----------------------------------------------------------------------
means, stdevs = {}, {}
for T, block in trials.items():
    means[T] = block.mean(axis=1)            # average of trials 1-3
    stdevs[T] = block.std(axis=1, ddof=1)    # ddof=1 -> sample std dev (n-1)

# ----------------------------------------------------------------------
# 4. Linear regression:  C_out = slope * tau + intercept
# ----------------------------------------------------------------------
# The fit uses every individual replicate rather than the averages so that the
# reported uncertainties reflect the true number of measurements. (With equal
# replicate counts the slope and intercept are identical either way.)
fits = {}
for T, block in trials.items():
    x = np.repeat(tau, block.shape[1])       # each tau appears once per trial
    y = block.ravel()                        # flatten trials into one vector
    fits[T] = stats.linregress(x, y)

# Print the coefficients to the console
print("Linear regression:  C_out = m * tau + b")
print("-" * 62)
for T, f in fits.items():
    print(f"T = {T} K")
    print(f"  slope     m = {f.slope: .5f} M/min   (+/- {f.stderr:.5f})")
    print(f"  intercept b = {f.intercept: .5f} M       (+/- {f.intercept_stderr:.5f})")
    print(f"  R^2         = {f.rvalue**2: .5f}")
    print()

In [ ]:

# ----------------------------------------------------------------------
# 5. Plot
# ----------------------------------------------------------------------
# Global style: sans-serif throughout, sizes chosen to stay legible in print.
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial", "Helvetica"],
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "legend.fontsize": 10,
    "axes.linewidth": 1.0,
})

fig, ax = plt.subplots(figsize=(6.5, 4.5))

colors = {300: "#1f77b4", 400: "#d62728"}
markers = {300: "o", 400: "s"}

# A smooth x-vector for drawing the fitted lines
tau_fit = np.linspace(tau.min(), tau.max(), 100)

# Handles are collected manually so the legend lists each data set
# immediately above its own fitted line.
handles = []

for T in TEMPERATURES:
    f = fits[T]

    # Measured points: mean of 3 trials, error bar = 1 standard deviation
    pts = ax.errorbar(
        tau, means[T], yerr=stdevs[T],
        fmt=markers[T], color=colors[T],
        markersize=6, markerfacecolor="white", markeredgewidth=1.4,
        capsize=4, elinewidth=1.2, linestyle="none",
        label=f"T = {T} K (mean $\\pm$ 1 s.d.)",
    )

    # Least-squares line, labelled with its own coefficients
    line, = ax.plot(
        tau_fit, f.slope * tau_fit + f.intercept,
        color=colors[T], linewidth=1.5, linestyle="--",
        label=(f"Fit: y = {f.slope:.4f}x + {f.intercept:.4f} "
               f"($R^2$ = {f.rvalue**2:.3f})"),
    )

    handles.extend([pts, line])

ax.set_title("Reactor concentration")
ax.set_xlabel("Residence time [min]")
ax.set_ylabel("Outlet Concentration [M]")

ax.set_xlim(0, tau.max() * 1.05)
ax.set_ylim(bottom=0)
ax.legend(handles=handles, frameon=False, loc="lower left")

# Light grid and clean spines are the usual journal convention
ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.6)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig("./"+OUT_PATH, format="jpeg", dpi=300)
print(f"Figure saved to {OUT_PATH}")